# TensorFlow: Build a multi-output model

Create an example of a multi-output model using functional API of TensorFlow.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

import tensorflow as tf

print("TF Version: ", tf.__version__)
print("TF Eager mode: ", tf.executing_eagerly())
print("TF GPU is", "available" if tf.config.list_physical_devices("GPU") else "not available")

# Prepare Dataset

__Download dataset__

In [ ]:
URL = "https://archive.ics.uci.edu/static/public/242/energy+efficiency.zip"

dir_path = tf.keras.utils.get_file(origin=URL, extract=True)

__Load dataset__

In [ ]:
df = pd.read_excel(os.path.join(dir_path, "ENB2012_data.xlsx"))
df = df.sample(frac=1).reset_index(drop=True)

__Prepare train and test datasets__

In [ ]:
def normalize(dataset, stats):
    return (dataset - stats["mean"]) / stats["std"]

def extract_y(dataset):
    y1 = dataset.pop("Y1")
    y2 = dataset.pop("Y2")
    return y1, y2

In [ ]:
train, test = train_test_split(df, test_size=0.2)

In [ ]:
# Calculate descriptive statistics (mean and std)
train_stats = train.describe()
train_stats.pop("Y1")
train_stats.pop("Y2")
train_stats = train_stats.transpose()

In [ ]:
# Create training dataset
train_y = extract_y(train)
train_x = normalize(train, train_stats)

# Create test dataset
test_y = extract_y(test)
test_x = normalize(test, train_stats)

# Build Model

__Create model__

In [ ]:
input = tf.keras.Input(shape=(len(train_x.columns),))
x1 = tf.keras.layers.Dense(units=128, activation=tf.nn.relu)(input)
x2 = tf.keras.layers.Dense(units=128, activation=tf.nn.relu)(x1)
output1 = tf.keras.layers.Dense(units=1, name="output1")(x2)
x3 = tf.keras.layers.Dense(units=64, activation=tf.nn.relu)(x2)
output2 = tf.keras.layers.Dense(units=1, name="output2")(x3)
model = tf.keras.Model(inputs=input, outputs=[output1, output2])

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
    loss={"output1": "mse",
          "output2": "mse"},
    metrics={"output1": tf.keras.metrics.RootMeanSquaredError(),
             "output2": tf.keras.metrics.RootMeanSquaredError()},
)

In [ ]:
tf.keras.utils.plot_model(model, show_shapes=True, show_layer_names=True)

__Train model__

In [ ]:
history = model.fit(
    train_x,
    train_y,
    epochs=500,
    batch_size=10,
    validation_data=(test_x, test_y))

__Evaluate model__

In [ ]:
def plot_metrics(history, metric_name, title, ylim=5):
    plt.title(title)
    plt.ylim(0, ylim)
    plt.plot(history[metric_name], color="blue", label=metric_name)
    plt.plot(history["val_" + metric_name], color="green", label="val_" + metric_name)
    plt.show()

In [ ]:
def plot_diff(y_true, y_pred, title=''):
    plt.scatter(y_true, y_pred)
    plt.title(title)
    plt.xlabel("True Values")
    plt.ylabel("Predictions")
    plt.axis("equal")
    plt.axis("square")
    plt.xlim(plt.xlim())
    plt.ylim(plt.ylim())
    plt.plot([-100, 100], [-100, 100])
    plt.show()

In [ ]:
loss, y1_loss, y2_loss, y1_rmse, y2_rmse = model.evaluate(x=test_x, y=test_y)

In [ ]:
y_pred = model.predict(x=test_x)

In [ ]:
plot_diff(test_y[0], y_pred[0], title="Y1")
plot_diff(test_y[1], y_pred[1], title="Y2")

In [ ]:
plot_metrics(history.history, metric_name='output1_root_mean_squared_error', title='Y1 RMSE', ylim=6)

In [ ]:
plot_metrics(history.history, metric_name='output2_root_mean_squared_error', title='Y2 RMSE', ylim=7)